<a href="https://colab.research.google.com/github/humaaslam46/Internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

## 1. My lane as an ML task (type)

This is a **scoring/ranking task**, built on top of a binary classifier. My lane
(Refresh / Content Opportunity Scoring) produces an ordered queue of pages ranked
by predicted decline probability — not a single yes/no label. A reviewer works
top-down through that queue, so what matters is the ordering, not just whether
any one prediction is right in isolation.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [4]:
task_type = "scoring/ranking (implemented via a classifier's predicted probability)"
print("ML task type:", task_type)

ML task type: scoring/ranking (implemented via a classifier's predicted probability)


## 2. Target or proxy

## 2. Target or proxy

The proxy label is `is_declining_label = trend_direction == "down"`. This comes
from a defined rule on the current window, not an observed future outcome — it's
a beginner proxy. A stronger version for later weeks: prior 90-day features
predicting next-30-day decline, which would make the label a true future
outcome instead of a current-state bucket.

In [5]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.isdir("Internship-ml"):
        os.system("git clone --depth 1 https://github.com/humaaslam46/Internship-ml")
    os.chdir("Internship-ml")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check you're at repo root"
print("Ready.")

Working dir: /content/Internship-ml/Internship-ml
Ready.


In [6]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True))

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

**Precision@50.** With limited reviewer capacity to check roughly 50 pages a
cycle, what matters is how many of the top 50 ranked pages are actually
declining — not overall accuracy across all 30,000 pages, which would be
dominated by the easy majority class and say nothing about review quality.

In [7]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

y = df["is_declining_label"].values
print("Baseline hand-rule Precision@50:", round(precision_at_k(df["hand_rule_score"], y, 50), 3))

Baseline hand-rule Precision@50: 0.68


## 4. The unit of analysis, as a real dataframe

One row = one content page (`content_id`), measured at a fixed 90-day snapshot.

In [8]:
lane_slice = df[["content_id", "client_id", "impressions_90d", "days_since_last_update",
                  "avg_position", "ctr", "trend_direction", "word_count",
                  "is_declining_label"]].copy()
lane_slice.head()

,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,word_count,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,down,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,down,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,down,3515.0,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,stable,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,down,2803.0,1


## 5. Why ML beats a fixed rule here

The starter pipeline's own verified results show a fixed baseline rule scoring
Precision@50 = 0.240, while a random forest reaches 0.740 — roughly 3x more
correct pages in the same top-50 slots. A hand rule like "stale AND visible"
can only combine 2-3 signals with hard cutoffs; it can't weigh six-plus signals
against each other or find non-obvious combinations the way a model can.

In [9]:
print("Documented baseline rule Precision@50: 0.240")
print("Documented random forest Precision@50: 0.740")
print("Improvement:", round(0.740 / 0.240, 2), "x")

Documented baseline rule Precision@50: 0.240
Documented random forest Precision@50: 0.740
Improvement: 3.08 x


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.